# RAM Antivirus — Member 2: Network Attacks & ML Pipeline
### **Assigned Scope:** C6 Data Exfiltration | C7 Network Scanning | C8 DDoS | C9 Cryptomining
**Author:** Member 2 — Network Security & Machine Learning  
**Supporting Engines:** `backend/engines/ml_engine.py`, `feature_extractor.py`, `train_model.py`

---
## Overview & Objectives
Member 2 is responsible for network attack detection and the PyOD unsupervised machine learning pipeline on the 9-dimensional operational network flow schema.


In [1]:
# %pip install -q joblib scikit-learn seaborn matplotlib pandas numpy pyod
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from pyod.models.iforest import IForest
from pyod.models.hbos import HBOS
from pyod.models.knn import KNN

sns.set_theme(style="whitegrid")
print("[OK] Member 2 ML environment ready!")


[OK] Member 2 ML environment ready!


---
## 1. The 9 Operational Network Flow Features (IEEE Specification)
Selected strictly from observable network packet headers to eliminate target leakage:
1. `duration`: Length of connection in seconds
2. `src_bytes`: Bytes sent from client to server
3. `dst_bytes`: Bytes sent from server to client
4. `count`: Connections to same host in past 2s
5. `srv_count`: Connections to same port in past 2s
6. `serror_rate`: Ratio of connections with SYN errors
7. `rerror_rate`: Ratio of connections with REJ errors
8. `same_srv_rate`: Ratio of connections to same service
9. `diff_srv_rate`: Ratio of connections to different services


In [2]:
FLOW_FEATURES = [
    "duration", "src_bytes", "dst_bytes", "count", "srv_count",
    "serror_rate", "rerror_rate", "same_srv_rate", "diff_srv_rate"
]

DATA_DIR = os.path.join("..", "backend", "data")
df_train = pd.read_csv(os.path.join(DATA_DIR, "KDDTrain+.txt"), header=None)

# Assign flow feature indices from KDD
# [0: duration, 4: src_bytes, 5: dst_bytes, 22: count, 23: srv_count, 24: serror_rate, 26: rerror_rate, 28: same_srv_rate, 29: diff_srv_rate]
X_raw = df_train[[0, 4, 5, 22, 23, 24, 26, 28, 29]].values
y_true = (df_train[41] != "normal").astype(int).values

print(f"9D Flow Feature Matrix: {X_raw.shape}")
print(f"Contamination Rate:     {np.mean(y_true)*100:.2f}%")


9D Flow Feature Matrix: (125973, 9)
Contamination Rate:     46.54%


---
## 2. Model Training & Benchmark Evaluation (Isolation Forest, HBOS, KNN)


In [3]:
# Scaling strictly on train data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Load test data
df_test = pd.read_csv(os.path.join(DATA_DIR, "KDDTest+.txt"), header=None)
X_test = scaler.transform(df_test[[0, 4, 5, 22, 23, 24, 26, 28, 29]].values)
y_test = (df_test[41] != "normal").astype(int).values

contamination = float(np.mean(y_true))

# Train PyOD Models
models = {
    "Isolation Forest": IForest(n_estimators=100, contamination=contamination, random_state=42),
    "HBOS": HBOS(n_bins=20, contamination=contamination),
    "KNN": KNN(n_neighbors=5, contamination=contamination)
}

results = {}
for name, clf in models.items():
    if name == "KNN":
        sub = np.random.RandomState(42).choice(len(X_scaled), size=10000, replace=False)
        clf.fit(X_scaled[sub])
    else:
        clf.fit(X_scaled)
    
    y_pred = (clf.decision_function(X_test) > clf.threshold_).astype(int)
    results[name] = {
        "Accuracy": f"{accuracy_score(y_test, y_pred)*100:.2f}%",
        "Precision": f"{precision_score(y_test, y_pred, zero_division=0)*100:.2f}%",
        "Recall": f"{recall_score(y_test, y_pred, zero_division=0)*100:.2f}%",
        "F1-Score": f"{f1_score(y_test, y_pred, zero_division=0)*100:.2f}%"
    }

pd.DataFrame(results).T


,Accuracy,Precision,Recall,F1-Score
Isolation Forest,77.51%,85.21%,73.19%,78.74%
HBOS,70.68%,85.30%,58.59%,69.47%
KNN,74.10%,85.60%,65.52%,74.23%


---
## 3. Assigned Detectors (C6, C7, C8, C9) Mechanics
- **C6 Data Exfiltration:** Checks `dst_bytes >= 100 MB / 500 MB` outbound to non-internal IP.
- **C7 Network Scanning:** Checks `diff_srv_rate >= 0.65` and `count >= 8` (port sweeps).
- **C8 DDoS:** Checks `count >= 150` or SYN flood `serror_rate >= 0.75`.
- **C9 Cryptomining:** Detects Stratum mining protocol ports (`3333`, `4444`, `8333`) and binaries (`xmrig`, `minerd`).


In [4]:
# Example rule logic check
def test_ddos(count, serror_rate):
    return "CRITICAL DDOS ALERT" if (count >= 150 or (serror_rate >= 0.75 and count >= 20)) else "Normal"

print("Test Flow (count=200, serror=0.85):", test_ddos(200, 0.85))
print("Test Flow (count=5,   serror=0.00):", test_ddos(5, 0.0))


Test Flow (count=200, serror=0.85): CRITICAL DDOS ALERT
Test Flow (count=5,   serror=0.00): Normal


---
## 4. Network & ML Architecture Engineering Reference
- **Q1: Why unsupervised ML?**  
  *Ans:* Zero-day attacks have no known signatures; unsupervised models learn normal traffic and flag deviations.
- **Q2: Why PyOD?**  
  *Ans:* PyOD is the industry benchmark library providing unified APIs and score normalization for outlier detection.
- **Q3: Does ML detect all 16 attacks?**  
  *Ans:* No. We use hybrid intelligence: deterministic detectors for known signatures, PyOD ML for 9D network flow anomalies.
